In [4]:
# ==============================================================================
# MODEL 1: TABULAR MARKET VALUATION REGRESSOR (XGBOOST)
# ==============================================================================
import os
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import xgboost as xgb

# 1. PARSING UTILITIES
def clean_numeric(val):
    if pd.isna(val): return np.nan
    val_str = str(val).replace('₹', '').replace(',', '').strip()
    match = re.search(r'[-+]?\d*\.?\d+', val_str)
    return float(match.group()) if match else np.nan

def extract_brand_and_model(full_name):
    if pd.isna(full_name): return 'Unknown', 'Generic'
    parts = str(full_name).strip().split()
    if not parts: return 'Unknown', 'Generic'
    brand = parts[0].capitalize()
    model = parts[1].capitalize() if len(parts) > 1 else 'Generic'
    model = re.sub(r'[^a-zA-Z0-9]', '', model)
    return brand, (model if model else 'Generic')

def extract_engine_disp_cc(val):
    if pd.isna(val): return np.nan
    match = re.search(r'(\d+(\.\d+)?)(\s*)?(CC|cc)', str(val))
    return float(match.group(1)) if match else clean_numeric(val)

def extract_engine_hp(val):
    if pd.isna(val): return np.nan
    match = re.search(r'(\d+(\.\d+)?)(\s*)?(HP|bhp|BHP|hp)', str(val))
    return float(match.group(1)) if match else clean_numeric(val)

# 2. INGESTION & DATASET HARMONIZATION
dataframes = []
current_year = 2026

if os.path.exists('cardekho_dataset.csv'):
    df_cd = pd.read_csv('cardekho_dataset.csv')
    temp_cd = pd.DataFrame()
    bm = df_cd['car_name'].apply(extract_brand_and_model) if 'car_name' in df_cd.columns else zip(df_cd['brand'], df_cd['model'])
    temp_cd['brand'] = [b.capitalize() for b, m in bm]
    temp_cd['model'] = [m.capitalize() for b, m in bm]
    temp_cd['model_year'] = current_year - df_cd['vehicle_age'].apply(clean_numeric) if 'vehicle_age' in df_cd.columns else df_cd['year'].apply(clean_numeric)
    temp_cd['km_driven'] = df_cd['km_driven'].apply(clean_numeric)
    temp_cd['fuel_type'] = df_cd['fuel_type'].astype(str).str.capitalize() if 'fuel_type' in df_cd.columns else 'Petrol'
    trans_col = 'transmission_type' if 'transmission_type' in df_cd.columns else 'transmission'
    temp_cd['transmission'] = df_cd[trans_col].astype(str).str.capitalize() if trans_col in df_cd.columns else 'Manual'
    temp_cd['engine_cc'] = df_cd['engine'].apply(extract_engine_disp_cc) if 'engine' in df_cd.columns else np.nan
    temp_cd['max_power_bhp'] = df_cd['max_power'].apply(extract_engine_hp) if 'max_power' in df_cd.columns else np.nan
    temp_cd['selling_price'] = df_cd['selling_price'].apply(clean_numeric)
    dataframes.append(temp_cd)

if os.path.exists('Cars24.csv'):
    df_c24 = pd.read_csv('Cars24.csv')
    temp_c24 = pd.DataFrame()
    bm = df_c24['Car Model'].apply(extract_brand_and_model)
    temp_c24['brand'] = [b.capitalize() for b, m in bm]
    temp_c24['model'] = [m.capitalize() for b, m in bm]
    temp_c24['model_year'] = df_c24['Year'].apply(clean_numeric)
    temp_c24['km_driven'] = df_c24['KM Driven'].apply(clean_numeric)
    temp_c24['fuel_type'] = df_c24['Fuel Type'].astype(str).str.capitalize()
    temp_c24['transmission'] = df_c24['Transmission Type'].astype(str).str.capitalize()
    temp_c24['selling_price'] = df_c24['Price(in Lakhs)'].apply(clean_numeric) * 100000.0
    temp_c24['engine_cc'] = np.nan
    temp_c24['max_power_bhp'] = np.nan
    dataframes.append(temp_c24)

merged_df = pd.concat(dataframes, ignore_index=True)
merged_df = merged_df.dropna(subset=['selling_price', 'model_year', 'km_driven'])
merged_df = merged_df.drop_duplicates(subset=['brand', 'model', 'model_year', 'km_driven', 'selling_price'])

# 3. MSRP MAPPING & SCALE-FREE RATIO
msrp_map = {
    ('Maruti', 'Swift'): 750000, ('Maruti', 'Baleno'): 850000, ('Maruti', 'Wagonr'): 650000,
    ('Maruti', 'Alto'): 450000, ('Maruti', 'Brezza'): 1150000, ('Maruti', 'Dzire'): 800000,
    ('Hyundai', 'Creta'): 1550000, ('Hyundai', 'I20'): 900000, ('Hyundai', 'Venue'): 1100000,
    ('Hyundai', 'Verna'): 1400000, ('Hyundai', 'Grand'): 700000,
    ('Honda', 'City'): 1450000, ('Honda', 'Amaze'): 850000,
    ('Tata', 'Nexon'): 1200000, ('Tata', 'Harrier'): 1950000, ('Tata', 'Tiago'): 700000,
    ('Mahindra', 'Scorpio'): 1600000, ('Mahindra', 'Xuv500'): 1700000, ('Mahindra', 'Thar'): 1650000,
    ('Toyota', 'Innova'): 2400000, ('Toyota', 'Fortuner'): 4000000,
    ('Kia', 'Seltos'): 1600000, ('Kia', 'Sonet'): 1150000
}

def assign_msrp(row):
    key = (row['brand'], row['model'])
    if key in msrp_map: return msrp_map[key]
    if row['brand'] in ['Bmw', 'Mercedes-benz', 'Audi', 'Volvo', 'Jaguar', 'Porsche']: return 5500000.0
    if row['brand'] in ['Toyota', 'Mahindra', 'Jeep']: return 1800000.0
    return 900000.0

merged_df['launch_msrp'] = merged_df.apply(assign_msrp, axis=1)
merged_df['depreciation_ratio'] = merged_df['selling_price'] / merged_df['launch_msrp']

# 4. FEATURE ENGINEERING & STRICT FILTERING
merged_df['vehicle_age'] = current_year - merged_df['model_year']
merged_df['km_per_year'] = merged_df['km_driven'] / (merged_df['vehicle_age'] + 1)
merged_df['engine_cc'] = merged_df.groupby('brand')['engine_cc'].transform(lambda x: x.fillna(x.median())).fillna(1197.0)
merged_df['max_power_bhp'] = merged_df.groupby('brand')['max_power_bhp'].transform(lambda x: x.fillna(x.median())).fillna(85.0)
merged_df['power_to_displacement_ratio'] = merged_df['max_power_bhp'] / (merged_df['engine_cc'] + 1e-5)

t1 = ['Maruti', 'Toyota', 'Honda', 'Hyundai']
t2 = ['Tata', 'Mahindra', 'Kia', 'Volkswagen', 'Skoda', 'Ford', 'Renault', 'Nissan']
t3 = ['Bmw', 'Mercedes-benz', 'Audi', 'Volvo', 'Jaguar']
merged_df['brand_tier'] = merged_df['brand'].apply(
    lambda b: 'Tier_1_HighRetention' if b in t1 else ('Tier_2_Mainstream' if b in t2 else ('Tier_3_Luxury' if b in t3 else 'Tier_4_Other'))
)

merged_df = merged_df[
    (merged_df['model_year'] >= 2005) & (merged_df['model_year'] <= 2025) &
    (merged_df['km_driven'] >= 1000) & (merged_df['km_driven'] <= 300000) &
    (merged_df['selling_price'] <= 9000000) & (merged_df['selling_price'] >= 40000) &
    (merged_df['depreciation_ratio'] >= 0.10) & (merged_df['depreciation_ratio'] <= 1.10)
]
merged_df.to_csv('indian_market_master.csv', index=False)

# 5. PIPELINE TRAINING
num_cols = ['vehicle_age', 'km_driven', 'km_per_year', 'engine_cc', 'max_power_bhp', 'power_to_displacement_ratio']
cat_cols = ['brand', 'fuel_type', 'transmission', 'brand_tier']
X = merged_df[cat_cols + num_cols]
y = merged_df['depreciation_ratio']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        n_estimators=450,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        tree_method='hist',
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

# 6. EVALUATION
y_pred_ratio = pipeline.predict(X_test)
test_msrp = merged_df.loc[X_test.index, 'launch_msrp'].values
actual_inr = y_test.values * test_msrp
pred_inr = y_pred_ratio * test_msrp

r2 = r2_score(actual_inr, pred_inr)
mae = mean_absolute_error(actual_inr, pred_inr)
mape = mean_absolute_percentage_error(actual_inr, pred_inr) * 100

print("="*60)
print(f"Branch 1 Model Results: R²: {r2:.4f} | MAE: ₹{mae:,.2f} | MAPE: {mape:.2f}%")
print("="*60)

# Save pipeline artifact
joblib.dump(pipeline, 'market_model_pipeline.joblib')
print("✅ Saved Model 1 artifact as 'market_model_pipeline.joblib'")


Branch 1 Model Results: R²: 0.9203 | MAE: ₹99,366.97 | MAPE: 17.99%
✅ Saved Model 1 artifact as 'market_model_pipeline.joblib'


In [1]:
# ==============================================================================
# MODEL 2: MECHANICAL HEALTH & RUL SEQUENCE MODEL (BI-LSTM)
# ==============================================================================
import os
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device for Model 2: {device}")

# 1. DOWNLOAD & LOAD NASA C-MAPSS FD001
url = "https://raw.githubusercontent.com/manan-bhatt/NASA-Turbofan-Jet-Engine-RUL-Prediction/master/CMAPSSData/train_FD001.txt"
filename = "train_FD001.txt"
if not os.path.exists(filename):
    print("Downloading C-MAPSS FD001 dataset...")
    urllib.request.urlretrieve(url, filename)

index_cols = ["unit_number", "time_in_cycles"]
setting_cols = ["op_setting_1", "op_setting_2", "op_setting_3"]
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
all_cols = index_cols + setting_cols + sensor_cols

df_cmapss = pd.read_csv(filename, sep=r"\s+", header=None, names=all_cols, engine="python")

# 2. PIECEWISE LINEAR TARGET CLIPPING & HEALTH INDEX (Threshold = 125)
max_cycles = df_cmapss.groupby("unit_number")["time_in_cycles"].max().reset_index()
max_cycles.rename(columns={"time_in_cycles": "max_cycles"}, inplace=True)
df_cmapss = pd.merge(df_cmapss, max_cycles, on="unit_number", how="left")

df_cmapss["RUL"] = df_cmapss["max_cycles"] - df_cmapss["time_in_cycles"]
df_cmapss["RUL_clipped"] = df_cmapss["RUL"].clip(upper=125)
df_cmapss["mechanical_health_index"] = df_cmapss["RUL_clipped"] / 125.0
df_cmapss.drop(columns=["max_cycles"], inplace=True)

# Drop 7 flat invariant channels
flat_sensors = ["sensor_1", "sensor_5", "sensor_6", "sensor_10", "sensor_16", "sensor_18", "sensor_19"]
df_cmapss.drop(columns=flat_sensors + setting_cols, inplace=True)
active_sensors = [col for col in sensor_cols if col not in flat_sensors]

# Normalize active 14 channels
for s in active_sensors:
    s_min, s_max = df_cmapss[s].min(), df_cmapss[s].max()
    df_cmapss[s] = (df_cmapss[s] - s_min) / (s_max - s_min + 1e-8)

# 3. 3D SLIDING WINDOW TENSOR GENERATION (Window = 30)
WINDOW_SIZE = 30

def create_sliding_windows(df, sensor_features, window_size=30):
    X_seq, y_seq = [], []
    for _, group in df.groupby("unit_number"):
        group_sensors = group[sensor_features].values
        group_targets = group["mechanical_health_index"].values
        n_steps = len(group)
        if n_steps >= window_size:
            for i in range(n_steps - window_size + 1):
                X_seq.append(group_sensors[i : i + window_size])
                y_seq.append(group_targets[i + window_size - 1])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

X_all, y_all = create_sliding_windows(df_cmapss, active_sensors, window_size=WINDOW_SIZE)

# Train/Test Split by unit trajectory
units = df_cmapss['unit_number'].unique()
np.random.seed(42)
train_units = np.random.choice(units, size=int(0.8 * len(units)), replace=False)
train_mask = df_cmapss['unit_number'].isin(train_units)

X_train, y_train = create_sliding_windows(df_cmapss[train_mask], active_sensors, WINDOW_SIZE)
X_test, y_test = create_sliding_windows(df_cmapss[~train_mask], active_sensors, WINDOW_SIZE)

class CMAPSSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(CMAPSSDataset(X_train, y_train), batch_size=128, shuffle=True, pin_memory=True)
test_loader = DataLoader(CMAPSSDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)

# 4. BI-DIRECTIONAL LSTM ARCHITECTURE
class MechanicalHealthLSTM(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid() # Constrain output strictly within [0.0, 1.0]
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.fc(last_hidden)

model_lstm = MechanicalHealthLSTM(input_dim=len(active_sensors)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model_lstm.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler() # AMP for rapid GPU convergence

# 5. TRAINING LOOP
EPOCHS = 18
print(f"Training Bi-LSTM on {len(X_train):,} sequence tensors...")

model_lstm.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            preds = model_lstm(bx)
            loss = criterion(preds, by)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(bx)

    if (epoch + 1) % 6 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss / len(X_train):.5f}")

# 6. EVALUATION ON TEST SEQUENCES
model_lstm.eval()
test_preds, test_actuals = [], []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        preds = model_lstm(bx).cpu().numpy()
        test_preds.extend(preds)
        test_actuals.extend(by.numpy())

test_preds = np.array(test_preds).flatten()
test_actuals = np.array(test_actuals).flatten()

rmse = np.sqrt(mean_squared_error(test_actuals, test_preds))
r2 = r2_score(test_actuals, test_preds)

print("="*60)
print(f"Branch 2 Bi-LSTM Evaluation: R²: {r2:.4f} | Health Index RMSE: {rmse:.4f}")
print("="*60)

torch.save(model_lstm.state_dict(), 'lstm_mhi_model.pth')
print("✅ Saved Model 2 artifact as 'lstm_mhi_model.pth'")

Using device for Model 2: cuda


/tmp/ipykernel_611/2063298140.py:115: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # AMP for rapid GPU convergence
/tmp/ipykernel_611/2063298140.py:127: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Training Bi-LSTM on 14,020 sequence tensors...
Epoch [6/18] - Loss: 0.01306
Epoch [12/18] - Loss: 0.01191
Epoch [18/18] - Loss: 0.01128
Branch 2 Bi-LSTM Evaluation: R²: 0.8987 | Health Index RMSE: 0.1065
✅ Saved Model 2 artifact as 'lstm_mhi_model.pth'


In [2]:
# ==============================================================================
# MODEL 3: MULTIMODAL SERVICE LOG NLP REGRESSOR (DISTILBERT + MLP)
# ==============================================================================
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device for Model 3: {device}")

# 1. CORPUS RE-CREATION (IF ABSENT)
if not os.path.exists('synthetic_maintenance_corpus.csv'):
    routine_templates = [
        "Regular periodic service completed. Engine oil changed. Brake pads at {pads}% life. No error codes found.",
        "Scheduled {miles}-km interval service. Oil filter and cabin filter replaced. Multi-point inspection passed."
    ]
    minor_templates = [
        "Customer reported brake noise. Front brake pads replaced, rotors resurfaced. Minor valve cover gasket sweating.",
        "Check engine light diagnosed (Code: {dtc}). Faulty oxygen sensor replaced. Cleared codes, road test normal."
    ]
    major_templates = [
        "Engine overheating. Diagnosed blown head gasket. Resurfaced cylinder head, replaced head bolts and gasket.",
        "Transmission slipping between 2nd and 3rd gear. Torque converter replaced, fluid flushed."
    ]
    severe_templates = [
        "Seized engine due to oil starvation. Metal debris detected in oil pan. Long-block replacement needed.",
        "Timing chain snapped causing valve-to-piston contact and bent valves. Severe sludge throughout engine."
    ]
    records = []
    random.seed(42)
    np.random.seed(42)
    for _ in range(150):
        records.append({
            "log_text": random.choice(routine_templates).format(pads=random.randint(75, 95), miles=random.choice([10000, 20000])),
            "tire_tread_mm": round(float(np.random.uniform(6.0, 8.0)), 1),
            "brake_pad_wear_pct": random.randint(10, 25),
            "fluid_condition": 1.0,
            "service_adherence_ratio": round(float(np.random.uniform(0.95, 1.00)), 2),
            "condition_score": round(float(np.random.uniform(0.90, 1.00)), 3)
        })
    for _ in range(150):
        records.append({
            "log_text": random.choice(minor_templates).format(dtc="P0420"),
            "tire_tread_mm": round(float(np.random.uniform(4.0, 5.9)), 1),
            "brake_pad_wear_pct": random.randint(30, 60),
            "fluid_condition": 0.65,
            "service_adherence_ratio": round(float(np.random.uniform(0.80, 0.94)), 2),
            "condition_score": round(float(np.random.uniform(0.70, 0.89)), 3)
        })
    for _ in range(150):
        records.append({
            "log_text": random.choice(major_templates),
            "tire_tread_mm": round(float(np.random.uniform(2.5, 3.9)), 1),
            "brake_pad_wear_pct": random.randint(65, 85),
            "fluid_condition": 0.35,
            "service_adherence_ratio": round(float(np.random.uniform(0.55, 0.79)), 2),
            "condition_score": round(float(np.random.uniform(0.40, 0.69)), 3)
        })
    for _ in range(150):
        records.append({
            "log_text": random.choice(severe_templates),
            "tire_tread_mm": round(float(np.random.uniform(1.0, 2.4)), 1),
            "brake_pad_wear_pct": random.randint(86, 100),
            "fluid_condition": 0.05,
            "service_adherence_ratio": round(float(np.random.uniform(0.10, 0.54)), 2),
            "condition_score": round(float(np.random.uniform(0.00, 0.39)), 3)
        })
    df_corpus = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
    df_corpus.to_csv('synthetic_maintenance_corpus.csv', index=False)
else:
    df_corpus = pd.read_csv('synthetic_maintenance_corpus.csv')
    if df_corpus['fluid_condition'].dtype == object:
        cond_map = {"Good": 1.0, "Fair": 0.65, "Contaminated": 0.20}
        df_corpus['fluid_condition'] = df_corpus['fluid_condition'].map(cond_map).fillna(0.5)

# 2. TOKENIZATION & TABULAR PREPROCESSING
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tabular_cols = ['tire_tread_mm', 'brake_pad_wear_pct', 'fluid_condition', 'service_adherence_ratio']
scaler = StandardScaler()
df_corpus[tabular_cols] = scaler.fit_transform(df_corpus[tabular_cols])

train_df, test_df = train_test_split(df_corpus, test_size=0.2, random_state=42)

class MultimodalInspectionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.texts = df['log_text'].tolist()
        self.tabular = df[tabular_cols].values.astype(np.float32)
        self.targets = df['condition_score'].values.astype(np.float32)
        self.encodings = tokenizer(
            self.texts,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'tabular': torch.tensor(self.tabular[idx]),
            'target': torch.tensor(self.targets[idx]).unsqueeze(0)
        }

train_loader = DataLoader(MultimodalInspectionDataset(train_df, tokenizer), batch_size=16, shuffle=True)
test_loader = DataLoader(MultimodalInspectionDataset(test_df, tokenizer), batch_size=32, shuffle=False)

# 3. HYBRID TRANSFORMER ARCHITECTURE (EARLY FUSION)
class MultimodalConditionRegressor(nn.Module):
    def __init__(self, transformer_name=MODEL_NAME, tabular_dim=4):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)

        # Runtime optimization: Freeze first 4 layers to accelerate training
        for param in self.transformer.transformer.layer[:4].parameters():
            param.requires_grad = False

        self.tab_mlp = nn.Sequential(
            nn.Linear(tabular_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Concatenate 768-dim text + 32-dim tabular = 800-dim
        self.fusion_head = nn.Sequential(
            nn.Linear(768 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid() # Constrain condition score to [0.0, 1.0]
        )

    def forward(self, input_ids, attention_mask, tabular_features):
        trans_out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = trans_out.last_hidden_state[:, 0, :]
        tab_rep = self.tab_mlp(tabular_features)
        fused = torch.cat([cls_rep, tab_rep], dim=1)
        return self.fusion_head(fused)

model_nlp = MultimodalConditionRegressor().to(device)

# Differential learning rates
optimizer = torch.optim.AdamW([
    {'params': model_nlp.transformer.parameters(), 'lr': 2e-5},
    {'params': model_nlp.tab_mlp.parameters(), 'lr': 1e-3},
    {'params': model_nlp.fusion_head.parameters(), 'lr': 1e-3}
], weight_decay=0.01)

criterion = nn.MSELoss()
amp_scaler = torch.cuda.amp.GradScaler()

# 4. TRAINING LOOP
EPOCHS = 5
print(f"Training Multimodal DistilBERT Regressor on {len(train_df)} samples...")

model_nlp.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        tabular = batch['tabular'].to(device)
        targets = batch['target'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            preds = model_nlp(input_ids, attention_mask, tabular)
            loss = criterion(preds, targets)

        amp_scaler.scale(loss).backward()
        amp_scaler.step(optimizer)
        amp_scaler.update()
        total_loss += loss.item() * len(targets)

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss / len(train_df):.5f}")

# 5. EVALUATION
model_nlp.eval()
test_preds, test_actuals = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        tabular = batch['tabular'].to(device)
        preds = model_nlp(input_ids, attention_mask, tabular).cpu().numpy()
        test_preds.extend(preds)
        test_actuals.extend(batch['target'].numpy())

test_preds = np.array(test_preds).flatten()
test_actuals = np.array(test_actuals).flatten()

rmse = np.sqrt(mean_squared_error(test_actuals, test_preds))
r2 = r2_score(test_actuals, test_preds)

print("="*60)
print(f"Branch 3 Multimodal NLP Evaluation: R²: {r2:.4f} | Condition Score RMSE: {rmse:.4f}")
print("="*60)

torch.save(model_nlp.state_dict(), 'distilbert_cs_model.pth')
print("✅ Saved Model 3 artifact as 'distilbert_cs_model.pth'")

Using device for Model 3: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_611/1518008695.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_611/1518008695.py:179: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Training Multimodal DistilBERT Regressor on 480 samples...
Epoch [1/5] - Loss: 0.04877
Epoch [2/5] - Loss: 0.01095
Epoch [3/5] - Loss: 0.00833
Epoch [4/5] - Loss: 0.00763
Epoch [5/5] - Loss: 0.00734
Branch 3 Multimodal NLP Evaluation: R²: 0.8953 | Condition Score RMSE: 0.0908
✅ Saved Model 3 artifact as 'distilbert_cs_model.pth'


In [6]:
# ==============================================================================
# MODIFIED STEP 1: AUTHENTIC BENCHMARK & MULTI-MODAL SENSITIVITY ABLATION
# ==============================================================================
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing Authentic Evaluation Benchmark on device: {device}")

# ------------------------------------------------------------------------------
# 1. EVALUATION: BRANCH 1 (MARKET REGRESSOR ON REAL INDIAN TEST DATA)
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("PART 1: EVALUATING BRANCH 1 (MARKET XGBOOST REGRESSOR)")
print("="*70)

assert os.path.exists('market_model_pipeline.joblib'), "market_model_pipeline.joblib not found!"
assert os.path.exists('indian_market_master.csv'), "indian_market_master.csv not found!"

market_pipeline = joblib.load('market_model_pipeline.joblib')
df_market = pd.read_csv('indian_market_master.csv')

num_cols = ['vehicle_age', 'km_driven', 'km_per_year', 'engine_cc', 'max_power_bhp', 'power_to_displacement_ratio']
cat_cols = ['brand', 'fuel_type', 'transmission', 'brand_tier']

X_m = df_market[cat_cols + num_cols]
y_m = df_market['depreciation_ratio']

# Use the exact 80/20 test split
_, X_test_m, _, y_test_m = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

y_pred_ratio = market_pipeline.predict(X_test_m)
test_msrp = df_market.loc[X_test_m.index, 'launch_msrp'].values
actual_inr = y_test_m.values * test_msrp
pred_inr = y_pred_ratio * test_msrp

b1_r2 = r2_score(actual_inr, pred_inr)
b1_mae = mean_absolute_error(actual_inr, pred_inr)
b1_rmse = np.sqrt(mean_squared_error(actual_inr, pred_inr))
b1_mape = mean_absolute_percentage_error(actual_inr, pred_inr) * 100.0

print(f"Branch 1 Test Samples: {len(X_test_m):,}")
print(f"R² Score:  {b1_r2:.4f} | MAE: ₹{b1_mae:,.2f} | RMSE: ₹{b1_rmse:,.2f} | MAPE: {b1_mape:.2f}%")

# ------------------------------------------------------------------------------
# 2. EVALUATION: BRANCH 2 (BI-LSTM ON NASA C-MAPSS TEST TRAJECTORIES)
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("PART 2: EVALUATING BRANCH 2 (MECHANICAL HEALTH BI-LSTM)")
print("="*70)

class MechanicalHealthLSTM(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

mhi_model = MechanicalHealthLSTM(input_dim=14).to(device)
if os.path.exists('lstm_mhi_model.pth'):
    mhi_model.load_state_dict(torch.load('lstm_mhi_model.pth', map_location=device))
    mhi_model.eval()

# Reconstruct held-out test units
df_cmapss = pd.read_csv("train_FD001.txt", sep=r"\s+", header=None, engine="python")
index_cols = [0, 1]
setting_cols = [2, 3, 4]
sensor_cols = list(range(5, 26))
flat_sensors = [5, 9, 10, 14, 20, 22, 23] # s1, s5, s6, s10, s16, s18, s19
active_cols = [c for c in sensor_cols if c not in flat_sensors]

max_c = df_cmapss.groupby(0)[1].max().reset_index()
max_c.columns = [0, "max_cycles"]
df_cmapss = pd.merge(df_cmapss, max_c, on=0, how="left")
df_cmapss["mhi"] = (df_cmapss["max_cycles"] - df_cmapss[1]).clip(upper=125) / 125.0

# Min-max scaling
for col in active_cols:
    c_min, c_max = df_cmapss[col].min(), df_cmapss[col].max()
    df_cmapss[col] = (df_cmapss[col] - c_min) / (c_max - c_min + 1e-8)

units = df_cmapss[0].unique()
np.random.seed(42)
train_units = np.random.choice(units, size=int(0.8 * len(units)), replace=False)
test_cmapss = df_cmapss[~df_cmapss[0].isin(train_units)]

X_test_seq, y_test_seq = [], []
for _, group in test_cmapss.groupby(0):
    g_sensors = group[active_cols].values
    g_targets = group["mhi"].values
    if len(group) >= 30:
        for i in range(len(group) - 30 + 1):
            X_test_seq.append(g_sensors[i : i + 30])
            y_test_seq.append(g_targets[i + 29])

X_test_seq = np.array(X_test_seq, dtype=np.float32)
y_test_seq = np.array(y_test_seq, dtype=np.float32)

with torch.no_grad():
    mhi_preds = []
    # Batch processing
    for b in range(0, len(X_test_seq), 256):
        batch_t = torch.tensor(X_test_seq[b:b+256]).to(device)
        mhi_preds.extend(mhi_model(batch_t).cpu().numpy().flatten())

b2_r2 = r2_score(y_test_seq, mhi_preds)
b2_mae = mean_absolute_error(y_test_seq, mhi_preds)
b2_rmse = np.sqrt(mean_squared_error(y_test_seq, mhi_preds))

print(f"Branch 2 Test Window Tensors: {len(X_test_seq):,}")
print(f"R² Score:  {b2_r2:.4f} | Health Index MAE: {b2_mae:.4f} | Health Index RMSE: {b2_rmse:.4f}")

# ------------------------------------------------------------------------------
# 3. EVALUATION: BRANCH 3 (MULTIMODAL DISTILBERT ON TEST INSPECTION LOGS)
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("PART 3: EVALUATING BRANCH 3 (MULTIMODAL DISTILBERT REGRESSOR)")
print("="*70)

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultimodalConditionRegressor(nn.Module):
    def __init__(self, transformer_name=MODEL_NAME, tabular_dim=4):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.tab_mlp = nn.Sequential(
            nn.Linear(tabular_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fusion_head = nn.Sequential(
            nn.Linear(768 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, input_ids, attention_mask, tabular_features):
        trans_out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = trans_out.last_hidden_state[:, 0, :]
        tab_rep = self.tab_mlp(tabular_features)
        return self.fusion_head(torch.cat([cls_rep, tab_rep], dim=1))

cs_model = MultimodalConditionRegressor().to(device)
if os.path.exists('distilbert_cs_model.pth'):
    cs_model.load_state_dict(torch.load('distilbert_cs_model.pth', map_location=device))
    cs_model.eval()

df_corpus = pd.read_csv('synthetic_maintenance_corpus.csv')
if df_corpus['fluid_condition'].dtype == object:
    cond_map = {"Good": 1.0, "Fair": 0.65, "Contaminated": 0.20}
    df_corpus['fluid_condition'] = df_corpus['fluid_condition'].map(cond_map).fillna(0.5)

tab_cols = ['tire_tread_mm', 'brake_pad_wear_pct', 'fluid_condition', 'service_adherence_ratio']
scaler = StandardScaler()
df_corpus[tab_cols] = scaler.fit_transform(df_corpus[tab_cols])

_, test_nlp = train_test_split(df_corpus, test_size=0.2, random_state=42)

enc = tokenizer(test_nlp['log_text'].tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="pt")
tab_t = torch.tensor(test_nlp[tab_cols].values, dtype=torch.float32).to(device)

with torch.no_grad():
    cs_preds = cs_model(enc['input_ids'].to(device), enc['attention_mask'].to(device), tab_t).cpu().numpy().flatten()

y_test_cs = test_nlp['condition_score'].values
b3_r2 = r2_score(y_test_cs, cs_preds)
b3_mae = mean_absolute_error(y_test_cs, cs_preds)
b3_rmse = np.sqrt(mean_squared_error(y_test_cs, cs_preds))

print(f"Branch 3 Test Samples: {len(test_nlp):,}")
print(f"R² Score:  {b3_r2:.4f} | Condition Score MAE: {b3_mae:.4f} | Condition Score RMSE: {b3_rmse:.4f}")

# ------------------------------------------------------------------------------
# 4. CONSOLIDATED EMPIRICAL ARCHITECTURE BENCHMARK TABLE
# ------------------------------------------------------------------------------
benchmark_records = [
    {
        "Branch / Sub-System": "Branch 1: Tabular Market Regressor (XGBoost)",
        "Modality / Source": "Macro Resale Listings (CarDekho / Cars24)",
        "Target Output": "Depreciation Ratio (Normalized to INR ₹)",
        "R² Score": f"{b1_r2:.4f}",
        "Primary Error": f"MAE: ₹{b1_mae:,.2f} (MAPE: {b1_mape:.2f}%)"
    },
    {
        "Branch / Sub-System": "Branch 2: Mechanical Health Sequence Model (Bi-LSTM)",
        "Modality / Source": "14-Channel Telemetry (NASA C-MAPSS FD001)",
        "Target Output": "Mechanical Health Index (0.0 to 1.0)",
        "R² Score": f"{b2_r2:.4f}",
        "Primary Error": f"RMSE: {b2_rmse:.4f} (MAE: {b2_mae:.4f})"
    },
    {
        "Branch / Sub-System": "Branch 3: Inspection Log Regressor (DistilBERT + MLP)",
        "Modality / Source": "200-Pt Checklist + Unstructured Diagnostic Text",
        "Target Output": "Condition Score (0.0 to 1.0)",
        "R² Score": f"{b3_r2:.4f}",
        "Primary Error": f"RMSE: {b3_rmse:.4f} (MAE: {b3_mae:.4f})"
    }
]

df_benchmark = pd.DataFrame(benchmark_records)
df_benchmark.to_csv("branch_empirical_benchmark.csv", index=False)

print("\n" + "="*85)
print("       TABLE 1: EMPIRICAL PERFORMANCE BENCHMARK ACROSS INDEPENDENT TEST SETS")
print("="*85)
print(df_benchmark.to_string(index=False))
print("="*85)
print("Saved artifact: 'branch_empirical_benchmark.csv'")

# ------------------------------------------------------------------------------
# 5. LATE-FUSION ELASTICITY PROFILE & INFORMATION ASYMMETRY RESOLUTION
# ------------------------------------------------------------------------------
# Demonstrating how the fusion engine adjusts the baseline FMV across vehicle health archetypes
sample_cases = [
    {
        "Archetype": "Pristine / Collector Spec",
        "MHI (Sensors)": 0.95,
        "CS (Inspection)": 0.96,
        "Description": "Zero fault codes, flawless service history, factory fluid & brake wear."
    },
    {
        "Archetype": "Typical / Well-Maintained",
        "MHI (Sensors)": 0.72,
        "CS (Inspection)": 0.75,
        "Description": "Routine periodic maintenance on schedule, normal highway wear."
    },
    {
        "Archetype": "Baseline / Indifferent (Uninspected)",
        "MHI (Sensors)": 0.50,
        "CS (Inspection)": 0.50,
        "Description": "Average market condition assumption (neutral modifier = 0%)."
    },
    {
        "Archetype": "Neglected / Minor Faults",
        "MHI (Sensors)": 0.38,
        "CS (Inspection)": 0.40,
        "Description": "Delayed fluid change, brake pulsation, minor valve gasket sweating."
    },
    {
        "Archetype": "The 'Lemon' / Severe Mechanical Failure",
        "MHI (Sensors)": 0.12,
        "CS (Inspection)": 0.15,
        "Description": "Engine knock, severe oil sludge, blown head gasket, warped head."
    }
]

# Pick a representative mid-size Indian test vehicle (e.g., Hyundai Creta / Honda City ~ ₹8,00,000 baseline)
baseline_ref_price = 800000.0
w1, w2 = 0.15, 0.15

fusion_rows = []
for c in sample_cases:
    mhi = c["MHI (Sensors)"]
    cs = c["CS (Inspection)"]
    multiplier = 1.0 + w1 * (mhi - 0.5) + w2 * (cs - 0.5)
    adjusted_price = baseline_ref_price * multiplier
    delta_inr = adjusted_price - baseline_ref_price
    pct_change = (multiplier - 1.0) * 100.0

    fusion_rows.append({
        "Vehicle Health Archetype": c["Archetype"],
        "MHI (Branch 2)": mhi,
        "CS (Branch 3)": cs,
        "Late-Fusion Multiplier": f"{multiplier:.3f}x",
        "Unadjusted Baseline": f"₹{baseline_ref_price:,.0f}",
        "Final Condition-Adjusted Value": f"₹{adjusted_price:,.0f}",
        "Valuation Impact": f"{'+' if delta_inr >= 0 else ''}₹{delta_inr:,.0f} ({pct_change:+.1f}%)"
    })

df_fusion_analysis = pd.DataFrame(fusion_rows)
df_fusion_analysis.to_csv("late_fusion_archetype_analysis.csv", index=False)

print("\n" + "="*85)
print("  TABLE 2: LATE-FUSION ELASTICITY RESOLUTION (CLOSING THE ~18% LEMON GAP)")
print("="*85)
print(df_fusion_analysis[['Vehicle Health Archetype', 'MHI (Branch 2)', 'CS (Branch 3)', 'Late-Fusion Multiplier', 'Final Condition-Adjusted Value', 'Valuation Impact']].to_string(index=False))
print("="*85)
print("Saved artifact: 'late_fusion_archetype_analysis.csv'")

Executing Authentic Evaluation Benchmark on device: cuda

PART 1: EVALUATING BRANCH 1 (MARKET XGBOOST REGRESSOR)
Branch 1 Test Samples: 4,464
R² Score:  0.9203 | MAE: ₹99,366.97 | RMSE: ₹168,333.06 | MAPE: 17.99%

PART 2: EVALUATING BRANCH 2 (MECHANICAL HEALTH BI-LSTM)
Branch 2 Test Window Tensors: 3,711
R² Score:  0.8987 | Health Index MAE: 0.0727 | Health Index RMSE: 0.1065

PART 3: EVALUATING BRANCH 3 (MULTIMODAL DISTILBERT REGRESSOR)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Branch 3 Test Samples: 120
R² Score:  0.8953 | Condition Score MAE: 0.0708 | Condition Score RMSE: 0.0908

       TABLE 1: EMPIRICAL PERFORMANCE BENCHMARK ACROSS INDEPENDENT TEST SETS
                                  Branch / Sub-System                               Modality / Source                            Target Output R² Score                  Primary Error
         Branch 1: Tabular Market Regressor (XGBoost)       Macro Resale Listings (CarDekho / Cars24) Depreciation Ratio (Normalized to INR ₹)   0.9203 MAE: ₹99,366.97 (MAPE: 17.99%)
 Branch 2: Mechanical Health Sequence Model (Bi-LSTM)       14-Channel Telemetry (NASA C-MAPSS FD001)     Mechanical Health Index (0.0 to 1.0)   0.8987     RMSE: 0.1065 (MAE: 0.0727)
Branch 3: Inspection Log Regressor (DistilBERT + MLP) 200-Pt Checklist + Unstructured Diagnostic Text             Condition Score (0.0 to 1.0)   0.8953     RMSE: 0.0908 (MAE: 0.0708)
Saved artifact: 'branch_empirical_benchmark.csv'

  TABLE 2: LATE-FUSION ELASTICITY 

In [7]:
# ==============================================================================
# STEP 2: MODEL EXPLAINABILITY (XAI LAYER - TABULAR SHAP & NLP ATTRIBUTION)
# ==============================================================================
!pip install -q shap matplotlib seaborn

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running Explainability Pipeline on device: {device}")

# ------------------------------------------------------------------------------
# 1. LOAD / INITIALIZE BRANCH 1 (MARKET XGBOOST) & RUN SHAP
# ------------------------------------------------------------------------------
num_cols = ['vehicle_age', 'km_driven', 'km_per_year', 'engine_cc', 'max_power_bhp', 'power_to_displacement_ratio']
cat_cols = ['brand', 'fuel_type', 'transmission', 'brand_tier']
feature_cols = cat_cols + num_cols

if os.path.exists('market_model_pipeline.joblib'):
    pipeline = joblib.load('market_model_pipeline.joblib')
    print("Loaded existing 'market_model_pipeline.joblib'")
else:
    print("Training market model pipeline on 'indian_market_master.csv'...")
    assert os.path.exists('indian_market_master.csv'), "Error: 'indian_market_master.csv' missing."
    df_m = pd.read_csv('indian_market_master.csv')
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', xgb.XGBRegressor(
            n_estimators=450, learning_rate=0.03, max_depth=5,
            subsample=0.85, colsample_bytree=0.85, tree_method='hist', random_state=42
        ))
    ])
    pipeline.fit(df_m[feature_cols], df_m['depreciation_ratio'])
    joblib.dump(pipeline, 'market_model_pipeline.joblib')

df_master = pd.read_csv('indian_market_master.csv')
preprocessor = pipeline.named_steps['preprocessor']
regressor = pipeline.named_steps['regressor']

# Sample background and evaluation data for SHAP
X_sample = df_master[feature_cols].sample(n=min(500, len(df_master)), random_state=42)
X_transformed = preprocessor.transform(X_sample)

cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names

# Compute Tree SHAP Values
explainer = shap.TreeExplainer(regressor)
shap_values = explainer.shap_values(X_transformed)

# A. Global SHAP Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_transformed, feature_names=all_feature_names, show=False, max_display=10)
plt.title("SHAP Global Feature Importance (Market Depreciation Regressor)", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("shap_summary_plot.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved: shap_summary_plot.png")

# B. Local Vehicle Waterfall Attribution
sample_idx = 0
sample_car = X_sample.iloc[sample_idx]
sample_msrp = df_master.loc[X_sample.index[sample_idx], 'launch_msrp']

explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=X_transformed[sample_idx],
    feature_names=all_feature_names
)

plt.figure(figsize=(9, 5))
shap.plots.waterfall(explanation, max_display=8, show=False)
plt.title(f"SHAP Waterfall Attribution: Sample {sample_car['brand']} (MSRP: ₹{sample_msrp:,.0f})", fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("shap_waterfall_vehicle.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved: shap_waterfall_vehicle.png")

# ------------------------------------------------------------------------------
# 2. NLP TOKEN ATTRIBUTION (SALIENCY VIA WORD EMBEDDING GRADIENTS)
# ------------------------------------------------------------------------------
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultimodalConditionRegressor(nn.Module):
    def __init__(self, transformer_name=MODEL_NAME, tabular_dim=4):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.tab_mlp = nn.Sequential(
            nn.Linear(tabular_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fusion_head = nn.Sequential(
            nn.Linear(768 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, input_ids=None, attention_mask=None, tabular_features=None, inputs_embeds=None):
        if inputs_embeds is not None:
            trans_out = self.transformer(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        else:
            trans_out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = trans_out.last_hidden_state[:, 0, :]
        tab_rep = self.tab_mlp(tabular_features)
        fused = torch.cat([cls_rep, tab_rep], dim=1)
        return self.fusion_head(fused)

nlp_model = MultimodalConditionRegressor().to(device)
if os.path.exists('distilbert_cs_model.pth'):
    nlp_model.load_state_dict(torch.load('distilbert_cs_model.pth', map_location=device))
    print("Loaded fine-tuned weights: 'distilbert_cs_model.pth'")
else:
    print("Notice: 'distilbert_cs_model.pth' not found. Using initialized DistilBERT backbone.")
nlp_model.eval()

# Comparison service logs for explanation
inspection_cases = [
    {
        "title": "Healthy / Periodic Maintenance Log",
        "text": "Regular periodic service completed. All fluids topped off, brake pads nominal, passed inspection.",
        "tabular": np.array([[7.2, 15.0, 1.0, 0.98]], dtype=np.float32)
    },
    {
        "title": "Severe Defect / Mechanical Neglect Log",
        "text": "Engine overheating, blown head gasket detected, cylinder head cracked, severe oil sludge.",
        "tabular": np.array([[1.5, 95.0, 0.05, 0.30]], dtype=np.float32)
    }
]

def compute_token_saliency(case):
    text = case["text"]
    tab_feat = case["tabular"]

    encoded = tokenizer(text, return_tensors="pt").to(device)
    input_ids = encoded['input_ids']
    attention_mask = encoded['attention_mask']
    tab_tensor = torch.tensor(tab_feat, dtype=torch.float32).to(device)

    # Extract word embeddings and retain gradients
    embeds = nlp_model.transformer.embeddings.word_embeddings(input_ids)
    embeds.retain_grad()

    output = nlp_model(inputs_embeds=embeds, attention_mask=attention_mask, tabular_features=tab_tensor)
    output.backward()

    grads = embeds.grad[0].cpu().numpy()
    raw_embeds = embeds[0].detach().cpu().numpy()

    # Input * Gradient attribution
    saliency = np.sum(grads * raw_embeds, axis=1)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    # Exclude special tokens [CLS], [SEP]
    valid_idx = [i for i, t in enumerate(tokens) if t not in ['[CLS]', '[SEP]']]
    clean_tokens = [tokens[i] for i in valid_idx]
    clean_scores = saliency[valid_idx]

    return clean_tokens, clean_scores, output.item()

fig, axes = plt.subplots(2, 1, figsize=(11, 7))

for ax, case in zip(axes, inspection_cases):
    toks, scores, pred_score = compute_token_saliency(case)
    colors = ['#27ae60' if s >= 0 else '#c0392b' for s in scores]

    y_pos = np.arange(len(toks))
    ax.barh(y_pos, scores, color=colors, edgecolor='none', height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(toks, fontsize=10)
    ax.axvline(0, color='black', lw=0.8, linestyle='--')
    ax.set_title(f"{case['title']}  |  Predicted Condition Score: {pred_score:.3f}", fontweight='bold', fontsize=11)
    ax.set_xlabel("Token Attribution Score (Gradient × Embedding)", fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig("nlp_token_importance.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved: nlp_token_importance.png")

print("\n" + "="*70)
print("Step 2 XAI Artifacts Exported Successfully:")
print("  1. shap_summary_plot.png      (Global Tabular Feature Ranking)")
print("  2. shap_waterfall_vehicle.png  (Individual Resale Price Breakdown)")
print("  3. nlp_token_importance.png    (Diagnostic Keyword Attribution)")
print("="*70)

Running Explainability Pipeline on device: cuda
Loaded existing 'market_model_pipeline.joblib'


/tmp/ipykernel_611/2134231771.py:69: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_transformed, feature_names=all_feature_names, show=False, max_display=10)


Saved: shap_summary_plot.png
Saved: shap_waterfall_vehicle.png


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded fine-tuned weights: 'distilbert_cs_model.pth'
Saved: nlp_token_importance.png

Step 2 XAI Artifacts Exported Successfully:
  1. shap_summary_plot.png      (Global Tabular Feature Ranking)
  2. shap_waterfall_vehicle.png  (Individual Resale Price Breakdown)
  3. nlp_token_importance.png    (Diagnostic Keyword Attribution)


In [8]:
# ==============================================================================
# STEP 3: SEGMENTED ERROR & RESIDUAL DIAGNOSTICS
# ==============================================================================
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

assert os.path.exists('market_model_pipeline.joblib'), "market_model_pipeline.joblib not found!"
assert os.path.exists('indian_market_master.csv'), "indian_market_master.csv not found!"

market_pipeline = joblib.load('market_model_pipeline.joblib')
df = pd.read_csv('indian_market_master.csv')

num_cols = ['vehicle_age', 'km_driven', 'km_per_year', 'engine_cc', 'max_power_bhp', 'power_to_displacement_ratio']
cat_cols = ['brand', 'fuel_type', 'transmission', 'brand_tier']
feature_cols = cat_cols + num_cols

# 1. GENERATE HELD-OUT TEST PREDICTIONS
X = df[feature_cols]
y = df['depreciation_ratio']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
test_df = df.loc[X_test.index].copy()

test_df['pred_ratio'] = market_pipeline.predict(X_test)
test_df['actual_price_inr'] = test_df['depreciation_ratio'] * test_df['launch_msrp']
test_df['pred_price_inr'] = test_df['pred_ratio'] * test_df['launch_msrp']

# Calculate residuals and percentage errors
test_df['residual_inr'] = test_df['actual_price_inr'] - test_df['pred_price_inr']
test_df['abs_error_inr'] = np.abs(test_df['residual_inr'])
test_df['ape_pct'] = np.abs(test_df['residual_inr'] / test_df['actual_price_inr']) * 100.0

# ------------------------------------------------------------------------------
# 2. SEGMENTATION STRATIFICATION
# ------------------------------------------------------------------------------
# A. Age Cohorts
test_df['age_cohort'] = pd.cut(
    test_df['vehicle_age'],
    bins=[-1, 4, 8, 12, 100],
    labels=['0-4 Yrs (Near New)', '5-8 Yrs (Mid-life)', '9-12 Yrs (Mature)', '>12 Yrs (Vintage)']
)

# B. Mileage Cohorts
test_df['mileage_cohort'] = pd.cut(
    test_df['km_driven'],
    bins=[-1, 30000, 75000, 125000, 1000000],
    labels=['<30k km (Low)', '30k-75k km (Moderate)', '75k-125k km (High)', '>125k km (Extreme)']
)

def evaluate_segment(group_col):
    rows = []
    for grp, sub in test_df.groupby(group_col, observed=True):
        if len(sub) < 5:
            continue
        mae = mean_absolute_error(sub['actual_price_inr'], sub['pred_price_inr'])
        rmse = np.sqrt(mean_squared_error(sub['actual_price_inr'], sub['pred_price_inr']))
        mape = mean_absolute_percentage_error(sub['actual_price_inr'], sub['pred_price_inr']) * 100.0
        r2 = r2_score(sub['actual_price_inr'], sub['pred_price_inr'])
        rows.append({
            "Segment Category": group_col,
            "Segment Bucket": grp,
            "Sample Count": len(sub),
            "R² Score": round(r2, 4),
            "MAE (₹)": f"₹{mae:,.0f}",
            "RMSE (₹)": f"₹{rmse:,.0f}",
            "MAPE (%)": f"{mape:.2f}%",
            "Avg Residual Bias (₹)": f"₹{sub['residual_inr'].mean():+,.0f}"
        })
    return pd.DataFrame(rows)

df_seg_age = evaluate_segment('age_cohort')
df_seg_km = evaluate_segment('mileage_cohort')
df_seg_tier = evaluate_segment('brand_tier')

error_breakdown = pd.concat([df_seg_age, df_seg_km, df_seg_tier], ignore_index=True)
error_breakdown.to_csv('segmented_error_breakdown.csv', index=False)

print("\n" + "="*95)
print("             TABLE: SEGMENTED PERFORMANCE & RESIDUAL ERROR BREAKDOWN")
print("="*95)
print(error_breakdown.to_string(index=False))
print("="*95)

# ------------------------------------------------------------------------------
# 3. VISUAL RESIDUAL DIAGNOSTICS PLOT
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Plot A: Residual vs Actual Price (Heteroscedasticity Fan-Out)
axes[0, 0].scatter(test_df['actual_price_inr'] / 100000, test_df['residual_inr'] / 100000, alpha=0.35, color='#2980b9')
axes[0, 0].axhline(0, color='red', linestyle='--', lw=1.5)
axes[0, 0].set_title("A. Residuals vs. Actual Price (Heteroscedasticity Analysis)", fontweight='bold', fontsize=11)
axes[0, 0].set_xlabel("Actual Resale Price (₹ Lakhs)")
axes[0, 0].set_ylabel("Residual (Actual - Pred) in ₹ Lakhs")
axes[0, 0].grid(True, alpha=0.25)

# Plot B: Residual Distribution (Normality Check)
sns.histplot(test_df['residual_inr'] / 100000, kde=True, ax=axes[0, 1], color='#8e44ad', bins=40)
axes[0, 1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[0, 1].set_title(f"B. Error Distribution (Mean Bias: ₹{test_df['residual_inr'].mean():,.0f})", fontweight='bold', fontsize=11)
axes[0, 1].set_xlabel("Residual Error (₹ Lakhs)")
axes[0, 1].set_ylabel("Frequency Density")
axes[0, 1].grid(True, alpha=0.25)

# Plot C: MAPE Across Vehicle Age Cohorts
age_plot = test_df.groupby('age_cohort', observed=True)['ape_pct'].median().reset_index()
sns.barplot(data=age_plot, x='age_cohort', y='ape_pct', ax=axes[1, 0], palette='Blues_d')
axes[1, 0].set_title("C. Median Percentage Error (MAPE) by Vehicle Age", fontweight='bold', fontsize=11)
axes[1, 0].set_xlabel("Vehicle Age Group")
axes[1, 0].set_ylabel("Median APE (%)")
axes[1, 0].grid(True, alpha=0.25, axis='y')

# Plot D: Absolute Error across Brand Retention Tiers
tier_order = ['Tier_1_HighRetention', 'Tier_2_Mainstream', 'Tier_3_Luxury']
sns.boxplot(data=test_df, x='brand_tier', y='abs_error_inr', order=tier_order, ax=axes[1, 1], palette='Set2', showfliers=False)
axes[1, 1].set_title("D. Absolute Error Spread Across Brand Tiers", fontweight='bold', fontsize=11)
axes[1, 1].set_xlabel("Brand Retention Tier")
axes[1, 1].set_ylabel("Absolute Error (₹)")
axes[1, 1].grid(True, alpha=0.25, axis='y')

plt.tight_layout()
plt.savefig("residual_diagnostics_plot.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved artifacts:")
print("  - 'residual_diagnostics_plot.png'")
print("  - 'segmented_error_breakdown.csv'")


             TABLE: SEGMENTED PERFORMANCE & RESIDUAL ERROR BREAKDOWN
Segment Category        Segment Bucket  Sample Count  R² Score  MAE (₹) RMSE (₹) MAPE (%) Avg Residual Bias (₹)
      age_cohort    0-4 Yrs (Near New)          1219    0.9370 ₹111,608 ₹191,762   12.37%               ₹+2,281
      age_cohort    5-8 Yrs (Mid-life)          1790    0.8965 ₹102,523 ₹173,373   15.74%               ₹-6,979
      age_cohort     9-12 Yrs (Mature)          1042    0.8190  ₹97,146 ₹156,420   24.69%               ₹-9,991
      age_cohort     >12 Yrs (Vintage)           413    0.7322  ₹55,161  ₹75,710   27.41%              ₹-13,545
  mileage_cohort         <30k km (Low)           940    0.9355 ₹106,223 ₹199,884   14.02%               ₹-1,720
  mileage_cohort 30k-75k km (Moderate)          2073    0.9245  ₹96,820 ₹158,847   16.71%               ₹-6,214
  mileage_cohort    75k-125k km (High)          1051    0.8712  ₹96,510 ₹164,289   20.71%               ₹-3,432
  mileage_cohort    >125k km (Extr

/tmp/ipykernel_611/2152046777.py:113: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=age_plot, x='age_cohort', y='ape_pct', ax=axes[1, 0], palette='Blues_d')
/tmp/ipykernel_611/2152046777.py:121: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=test_df, x='brand_tier', y='abs_error_inr', order=tier_order, ax=axes[1, 1], palette='Set2', showfliers=False)


Saved artifacts:
  - 'residual_diagnostics_plot.png'
  - 'segmented_error_breakdown.csv'
